In [69]:
import pandas as pd
from pathlib import Path
from matplotlib import pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [70]:
ano = 2023

In [71]:
data_dir = Path("data")
INDIR = Path(f"../data/data_processed/{ano}")
OUTDIR_IMG = Path(f"../report/img/{ano}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [72]:
arquivo_municipio = INDIR / f"ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_TRATADO_{ano}.csv"
df = pd.read_csv(arquivo_municipio, sep=",")

In [73]:
df.head()

,COD_MUNICIPIO,UF,QTD_PARTICIPANTES,NOTA_CN_MEDIA,NOTA_CH_MEDIA,NOTA_LC_MEDIA,NOTA_MT_MEDIA,NOTA_REDACAO_MEDIA,NOTA_GERAL_MEDIA,MUNICIPIO,RENDA_FAMILIAR_SM_MEDIA
0,1100015,RO,244,482.932787,507.912705,499.709016,519.652869,654.016393,532.844754,ALTA FLORESTA D'OESTE,1.611017
1,1100023,RO,1434,487.027685,508.174477,504.455788,511.399930,615.034868,525.218550,ARIQUEMES,1.776483
2,1100049,RO,1571,492.579631,517.086442,509.535582,528.330554,607.218332,530.950108,CACOAL,1.906845
3,1100056,RO,228,480.655263,505.057895,502.089912,499.915351,617.807018,521.105088,CEREJEIRAS,1.916968
4,1100064,RO,269,495.703717,507.392565,500.521561,514.278067,603.420074,524.263197,COLORADO DO OESTE,2.139803


In [74]:
analise_notas = ['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA']

mapa_areas = {
    'CN': 'Ciências da Natureza e suas Tecnologias',
    'MT': 'Matemática e suas Tecnologias',
    'CH': 'Ciências Humanas e suas Tecnologias',
    'LC': 'Linguagens, Códigos e suas Tecnologias',
    'REDACAO': 'Redação'
}

# Paleta Okabe-Ito (color-blind friendly)
mapa_cores = {
    'CN': '#0072B2',      # azul
    'MT': '#E69F00',      # laranja
    'CH': '#009E73',      # verde
    'LC': '#D55E00',      # vermelho-alaranjado
    'REDACAO': '#CC79A7'  # roxo
}

for col in analise_notas:
    area_sigla = col.replace('NOTA_', '').replace('_MEDIA', '')
    area_nome = mapa_areas.get(area_sigla, area_sigla)
    cor_area = mapa_cores.get(area_sigla, '#0072B2')

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=('Histograma', 'Box Plot')
    )

    fig.add_trace(
        go.Histogram(
            x=df[col].round(2),
            nbinsx=200,
            histnorm='probability density',
            marker_color=cor_area,
            name=area_nome
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Box(y=df[col], marker_color=cor_area, name=area_nome),
        row=1, col=2
    )

    fig.update_layout(
        title_text=f'Distribuição de Notas - {area_nome}',
        showlegend=False,
        width=1200,
        height=450,
        template='plotly_white'
    )
    fig.show()

In [75]:
col_renda = 'RENDA_FAMILIAR_SM_MEDIA'
renda_2c = df[col_renda].round(2)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Histograma', 'Box Plot')
)

fig.add_trace(
    go.Histogram(
        x=renda_2c,
        nbinsx=200,
        histnorm='probability density',
        marker_color='#0072B2',
        name='Renda Familiar Média (SM)'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        y=renda_2c,
        marker_color='#0072B2',
        name='Renda Familiar Média (SM)'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Distribuição de Renda Familiar Média (SM)',
    showlegend=False,
    width=1200,
    height=450,
    template='plotly_white'
)

fig.show()

In [ ]:
candidatas_renda = ["RENDA_FAMILIAR_SM_MEDIA"]
candidatas_nota = ["NOTA_GERAL_MEDIA"]
candidatas_municipio = ["NO_MUNICIPIO", "MUNICIPIO", "NOME_MUNICIPIO"]
candidatas_uf = ["SG_UF", "UF", "SIGLA_UF"]

col_municipio = next((c for c in candidatas_municipio if c in df.columns), None)
col_uf = next((c for c in candidatas_uf if c in df.columns), None)

if col_municipio is None or col_uf is None:
    raise KeyError(
        "Colunas de municipio/UF nao encontradas. Disponiveis: "
        f"{list(df.columns)}"
    )

df_plot = df[[
    candidatas_renda[0],
    candidatas_nota[0],
    col_municipio,
    col_uf,
]].dropna()

correlacao = df_plot[candidatas_renda[0]].corr(df_plot[candidatas_nota[0]])

fig = go.Figure(
    data=go.Scatter(
        x=df_plot[candidatas_renda[0]],
        y=df_plot[candidatas_nota[0]],
        mode="markers",
        marker=dict(color="#0072B2", size=6),
        customdata=df_plot[[col_municipio, col_uf]].values,
        hovertemplate=(
            "Nome do Municipio: %{customdata[0]}<br>"
            "UF: %{customdata[1]}<br>"
            "Nota Geral Media: %{y:.2f}<br>"
            "Renda Familiar Media (SM): %{x:.2f}<extra></extra>"
        ),
        name="Municípios"
    )
)

fig.update_layout(
    title=f"Dispersao: Renda Familiar x Nota Geral Média (r = {correlacao:.3f})",
    xaxis_title="Renda Familiar Media (SM)",
    yaxis_title="Nota Geral Media",
    template="plotly_white",
    width=1200,
    height=450,
)

fig.show()